In [8]:
# multifile calibration

import cv2
import os
from dataloader import MOVE

In [35]:
caldir = r"C:\Users\aatid\Downloads\FILM\calibration\dimTH332"

COUNT = 0
DIST_SCALE = 0
PT1 = (0, 0)
PT2 = (0, 0)

def distance_calibration(event, x, y, flags, param):
    global DIST_SCALE
    global PT1
    global PT2
    global COUNT
    if event == cv2.EVENT_LBUTTONDOWN and COUNT == 0:
        DIST_SCALE = y
        PT1 = (x, y)
        COUNT = 1
        cv2.destroyAllWindows()
    elif event == cv2.EVENT_LBUTTONDOWN and COUNT == 1:
        DIST_SCALE = abs(y - DIST_SCALE)
        PT2 = (x, y)
        COUNT = 0
        cv2.destroyAllWindows()
    return

In [36]:
scales = []

for file in os.listdir(caldir):
    vid = MOVE(os.path.join(caldir, file))

    for i in range(2):
        COUNT = 0 # do not change
        frame = vid.frames[5][200:].copy()

        cv2.imshow("Click top of marker or top of fin.", frame)
        cv2.setMouseCallback("Click top of marker or top of fin.", distance_calibration)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
        cv2.circle(frame, (PT1[0], PT1[1]), 2, (66, 245, 81), 3)
        #print(PT1)

        marker = input("Markers visible? [y / n] ")
        if marker != 'y':
            FULLFIN = True
            cv2.imshow("Click bottom of fin.", frame)
            cv2.setMouseCallback("Click bottom of fin.", distance_calibration)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            cv2.circle(frame, (PT2[0], PT2[1]), 2, (66, 245, 81), 3)
            #print(PT2)
        else:
            FULLFIN = False
            dists = []
            for i in range(4):
                cv2.imshow("Click top of next marker.", frame)
                cv2.setMouseCallback("Click top of next marker.", distance_calibration)
                cv2.waitKey(0)
                cv2.destroyAllWindows()
                if i % 2 == 0:
                    cv2.circle(frame, (PT2[0], PT2[1]), 2, (66, 245, 81), 3)
                else:
                    cv2.circle(frame, (PT1[0], PT1[1]), 2, (66, 245, 81), 3)
                dists.append(abs(PT1[1] - PT2[1]))
            DIST_SCALE = sum(dists) / len(dists)

        # cv2.imshow("Marker location display.", frame)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()

        if FULLFIN:
            px_mm_cal = DIST_SCALE / 100
        else:
            px_mm_cal = DIST_SCALE / 10
        print(px_mm_cal)
        scales.append(px_mm_cal)

3.93
3.93
3.91
3.93
3.93
3.93


In [37]:
cal = sum(scales) / len(scales)
print(cal)

3.926666666666667
